# RAG retrieval evaluation — recall@k

This notebook evaluates the retrieval layer (`apps/backend/src/zen_backend/services/retrieval.py`). For each of 30 hand-labeled queries — written from the personal stress themes that the daily generator actually conditions on (career uncertainty, placement pressure, comparison, future anxiety, etc.) — we record the set of "expected traditions" we'd be happy to see surface, then run pgvector retrieval and check whether any expected tradition appears in the top-k results.

**Recall@k** for a query is `1` if any expected tradition appears in the top-k retrieved passages, else `0`. We report mean recall@1 / @3 / @5 / @10. The implementation plan §1 success target is **recall@5 ≥ 0.80**.

The 30 seeded queries live at `assets/eval/rag_eval_queries.jsonl` and can be edited or extended. Each row has `id`, `query`, `stress_theme`, and `expected_traditions`. To get to 50 (the plan target), append your own queries.

**This notebook needs the live retrieval pipeline** — Gemini embeddings + pgvector — so it has to be run against a configured backend. The other three notebooks (bandit, corpus stats, faithfulness) are decoupled and run anywhere.

## Setup


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
QUERIES_PATH = ROOT / "assets" / "eval" / "rag_eval_queries.jsonl"
RESULTS_PATH = ROOT / "assets" / "eval" / "rag_eval_results.json"

# Add the backend src to sys.path so we can import retrieval directly.
sys.path.insert(0, str(ROOT / "apps" / "backend" / "src"))

# Load .env so SUPABASE_DB_DSN_POOLER and GEMINI_API_KEY are available.
try:
    from dotenv import load_dotenv

    load_dotenv(ROOT / ".env")
except ImportError:
    pass

queries: list[dict] = []
with QUERIES_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            queries.append(json.loads(line))

print(f"Loaded {len(queries)} hand-labeled queries from {QUERIES_PATH.relative_to(ROOT)}")
print(f"Stress themes covered: {sorted({q['stress_theme'] for q in queries})}")

## Run retrieval over each query

Cell below imports the production retrieval function and calls it once per query, asking for the top-10 passages. We keep the full per-query result list so we can recompute recall@k for any k without re-running retrieval.

If pgvector or Gemini isn't reachable from where this notebook runs, the cell will raise — that's fine, the framework is what we're shipping today; populate the results JSON when you next have credentials handy.


In [ ]:
def run_retrieval(force: bool = False, top_k: int = 10) -> dict:
    """Run live retrieval against pgvector for every query in the seed set.

    Returns a dict with `queries` (the list of {query, retrieved_traditions})
    and `meta` (timestamp, top_k, count). Saves the result to
    `assets/eval/rag_eval_results.json` so re-runs of recall computation
    don't need DB access.
    """
    if RESULTS_PATH.exists() and not force:
        existing = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
        print(f"Using cached results from {RESULTS_PATH.relative_to(ROOT)}")
        return existing

    # Importing here so the cell only fails when we actually attempt a live run.
    from zen_backend.services.retrieval import fetch_ranked_passages

    out: list[dict] = []
    for q in queries:
        try:
            ranked = fetch_ranked_passages(
                q["query"],
                source_tier="gold",
                required_tags=None,
                prefer_season_words=False,
                limit=top_k,
            )
            traditions = [str(p.get("tradition") or "") for p in ranked]
        except Exception as exc:  # noqa: BLE001
            print(f"  ! {q['id']}: retrieval failed ({exc})")
            traditions = []
        out.append(
            {
                "id": q["id"],
                "query": q["query"],
                "stress_theme": q["stress_theme"],
                "expected_traditions": q["expected_traditions"],
                "retrieved_traditions": traditions,
            }
        )

    from datetime import datetime, timezone

    payload = {
        "meta": {
            "computed_at": datetime.now(timezone.utc).isoformat(),
            "top_k": top_k,
            "count": len(out),
        },
        "queries": out,
    }
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    RESULTS_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Saved {len(out)} query results -> {RESULTS_PATH.relative_to(ROOT)}")
    return payload


# Try a real run; if it fails, fall back to a stub so the rest of the notebook
# still tells the reader what shape the results would take.
try:
    results = run_retrieval(force=False)
except Exception as exc:  # noqa: BLE001
    print(f"Live retrieval not available here: {exc}")
    print("\nFalling back to a stub run so you can preview the recall metric.")
    results = {
        "meta": {"computed_at": "n/a", "top_k": 10, "count": len(queries)},
        "queries": [
            {
                **q,
                "retrieved_traditions": q["expected_traditions"][:1] + ["aesop", "tagore"],
            }
            for q in queries
        ],
    }

In [ ]:
def recall_at_k(query_results: list[dict], k: int) -> float:
    if not query_results:
        return 0.0
    hits = 0
    for q in query_results:
        expected = set(q.get("expected_traditions", []))
        retrieved_top_k = q.get("retrieved_traditions", [])[:k]
        if expected & set(retrieved_top_k):
            hits += 1
    return hits / len(query_results)


ks = [1, 3, 5, 10]
recall_curve = {k: recall_at_k(results["queries"], k) for k in ks}
print("Recall@k:")
for k in ks:
    print(f"  recall@{k:<2} = {recall_curve[k]:.3f}")
print(f"\nTarget per implementation plan §1: recall@5 ≥ 0.80")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ks, [recall_curve[k] for k in ks], marker="o", color="#2f5b4f", linewidth=2)
ax.axhline(0.80, color="#cb9366", linestyle="--", label="target = 0.80")
ax.set_xlabel("k (top-k retrieved)")
ax.set_ylabel("recall@k")
ax.set_xticks(ks)
ax.set_ylim(0, 1.05)
ax.set_title(f"Retrieval recall@k over {len(queries)} hand-labeled queries")
ax.legend(frameon=False)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## Per-query inspection

A single low recall@5 query is more informative than the aggregate. Below we list every miss so the corpus / retrieval can be tuned where it actually fails.


In [ ]:
misses = []
for q in results["queries"]:
    expected = set(q["expected_traditions"])
    top5 = q["retrieved_traditions"][:5]
    if not expected & set(top5):
        misses.append(q)

print(f"recall@5 misses: {len(misses)} / {len(results['queries'])}\n")
for m in misses:
    print(f"  [{m['id']}] {m['query']!r}")
    print(f"     stress_theme:    {m['stress_theme']}")
    print(f"     expected:        {m['expected_traditions']}")
    print(f"     retrieved (top5):{m['retrieved_traditions'][:5]}")
    print()

## Extending to 50 queries

The plan target is 50 queries; the seed file ships with 30. To extend:

1. Open `assets/eval/rag_eval_queries.jsonl`.
2. Append rows in the same JSONL shape:
   ```json
   {"id":"q31","query":"…","stress_theme":"…","expected_traditions":["…","…"]}
   ```
3. Use stress themes from `config/personal.yaml:profile.stress_themes` to keep the eval grounded in the actual personalization signal.
4. Re-run this notebook with `run_retrieval(force=True)` to refresh `rag_eval_results.json`.

Aim for diverse coverage: at least 4 queries per major stress theme (career, comparison, future anxiety, day-load), at least 2 queries per available season (the corpus's nature traditions skew summer-heavy), and at least 5 evening / end-of-day queries.
